In [ ]:
# ==========================================
# Dual Encoder Multitask NLP Model
# MuRIL + BERT + ByteBPE + Late Fusion
# ==========================================

import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import sentencepiece as spm
import matplotlib.pyplot as plt

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModel,
    BertModel,
    get_cosine_schedule_with_warmup
)

# ==========================================
# DEVICE
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# ==========================================
# LOAD DATA
# ==========================================
def load_data(train_path, test_path):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    return train_df, test_df

# ==========================================
# LABEL ENCODING
# ==========================================
def encode_labels(train_df, test_df):
    label_encoders = {}

    for task in train_df['task'].unique():
        le = LabelEncoder()

        train_df.loc[train_df['task'] == task, 'label_enc'] = \
            le.fit_transform(train_df[train_df['task'] == task]['label'])

        test_df.loc[test_df['task'] == task, 'label_enc'] = \
            le.transform(test_df[test_df['task'] == task]['label'])

        label_encoders[task] = le

    return train_df, test_df, label_encoders

# ==========================================
# DATASET CLASS
# ==========================================
class MultiTaskDataset(Dataset):
    def __init__(self, df, tokenizer, sp_model):
        self.df = df
        self.tokenizer = tokenizer
        self.sp = sp_model

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        bpe_text = " ".join(self.sp.encode(row['text'], out_type=str))

        enc = self.tokenizer(
            bpe_text,
            truncation=True,
            padding='max_length',
            max_length=128,
            return_tensors='pt'
        )

        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label': torch.tensor(int(row['label_enc'])),
            'task': row['task']
        }

# ==========================================
# MODEL
# ==========================================
class DualEncoderModel(nn.Module):
    def __init__(self, task_to_num_classes, tokenizer_len):
        super().__init__()

        # MuRIL
        self.muril = AutoModel.from_pretrained("google/muril-base-cased")

        # Freeze most layers
        for name, param in self.muril.named_parameters():
            if "encoder.layer.11" not in name and "pooler" not in name:
                param.requires_grad = False

        # BERT
        self.bert = BertModel.from_pretrained("bert-base-multilingual-cased")
        self.bert.resize_token_embeddings(tokenizer_len)

        # Fusion Layer
        self.fusion = nn.Sequential(
            nn.Linear(1536, 1024),
            nn.ReLU(),
            nn.LayerNorm(1024),
            nn.Dropout(0.1),
            nn.Linear(1024, 768),
            nn.LayerNorm(768)
        )

        self.dropout = nn.Dropout(0.3)

        # Task-specific heads
        self.heads = nn.ModuleDict({
            task: nn.Linear(768, num_cls)
            for task, num_cls in task_to_num_classes.items()
        })

        # Learnable task weights
        self.task_weights = nn.ParameterDict({
            task: nn.Parameter(torch.tensor(0.0))
            for task in task_to_num_classes
        })

    def forward(self, input_ids, attention_mask, task):
        muril_cls = self.muril(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state[:, 0, :]

        bert_cls = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state[:, 0, :]

        combined = torch.cat([muril_cls, bert_cls], dim=1)
        fused = self.fusion(combined)

        return self.heads[task](fused)

# ==========================================
# TRAIN FUNCTION
# ==========================================
def train_model(model, train_loader, val_loader, criterions, optimizer, scheduler, task_weight, epochs=50):

    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            tasks = batch['task']

            optimizer.zero_grad()
            task_loss = 0

            for task in set(tasks):
                idxs = [i for i, t in enumerate(tasks) if t == task]

                inp = input_ids[idxs]
                mask = attention_mask[idxs]
                lbl = labels[idxs]

                logits = model(inp, mask, task)
                loss = criterions[task](logits, lbl)

                weight = model.task_weights[task]
                weighted_loss = task_weight[task] * (torch.exp(-weight) * loss + weight)

                task_loss += weighted_loss

            task_loss.backward()
            optimizer.step()
            scheduler.step()

            total_loss += task_loss.item()

        train_losses.append(total_loss)
        print(f"Epoch {epoch+1} Train Loss: {total_loss:.4f}")

        # Validation
        model.eval()
        val_loss = 0

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                tasks = batch['task']

                task_loss = 0

                for task in set(tasks):
                    idxs = [i for i, t in enumerate(tasks) if t == task]

                    inp = input_ids[idxs]
                    mask = attention_mask[idxs]
                    lbl = labels[idxs]

                    logits = model(inp, mask, task)
                    loss = criterions[task](logits, lbl)

                    weight = model.task_weights[task]
                    weighted_loss = task_weight[task] * (torch.exp(-weight) * loss + weight)

                    task_loss += weighted_loss

                val_loss += task_loss.item()

        val_losses.append(val_loss)
        print(f"Epoch {epoch+1} Val Loss: {val_loss:.4f}")

    return train_losses, val_losses

# ==========================================
# PLOT
# ==========================================
def plot_losses(train_losses, val_losses):
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.grid()
    plt.show()

# ==========================================
# MAIN
# ==========================================
def main():

    train_df, test_df = load_data(
        "/kaggle/input/working-files/final_multitask___train.csv",
        "/kaggle/input/working-files/final_multitask___test.csv"
    )

    train_df, test_df, label_encoders = encode_labels(train_df, test_df)

    # Tokenizer + SentencePiece
    sp = spm.SentencePieceProcessor()
    sp.load("/kaggle/input/final-muti-head-dataset/byte_bpe_model.model")

    tokenizer = AutoTokenizer.from_pretrained(
        "bert-base-multilingual-cased",
        vocab_file="/kaggle/input/working-files/vocab.txt",
        use_fast=False
    )

    train_loader = DataLoader(
        MultiTaskDataset(train_df, tokenizer, sp),
        batch_size=32,
        shuffle=True
    )

    val_loader = DataLoader(
        MultiTaskDataset(test_df, tokenizer, sp),
        batch_size=32
    )

    num_classes = {task: len(le.classes_) for task, le in label_encoders.items()}

    model = DualEncoderModel(num_classes, len(tokenizer)).to(device)

    # Loss
    criterions = {}
    task_weight = {'SA': 1.5, 'OFFENS': 1.7, 'OTHERCAT': 1.0}

    for task in label_encoders:
        y = train_df[train_df.task == task]['label_enc']
        weights = compute_class_weight('balanced', classes=np.unique(y), y=y)

        criterions[task] = nn.CrossEntropyLoss(
            weight=torch.tensor(weights, dtype=torch.float).to(device),
            label_smoothing=0.2
        )

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

    total_steps = len(train_loader) * 50
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=200,
        num_training_steps=total_steps
    )

    train_losses, val_losses = train_model(
        model, train_loader, val_loader,
        criterions, optimizer, scheduler,
        task_weight, epochs=50
    )

    plot_losses(train_losses, val_losses)


if __name__ == "__main__":
    main()